# Petite frequency — respiratory competence in PDH complex mutants

## Assay

Approximately 200 cells from mid-log-phase cultures, grown at the indicated
temperature and duration, were plated onto YPG supplemented with 0.1% glucose.
Colonies were scored after 2–3 days at 30 °C as either large
(respiratory-competent, rho⁺) or small (petite, rho⁰/rho⁻).

The limiting glucose allows petite cells to form small colonies rather than none
at all, so both classes are countable on the same plate and the readout is a
size distribution rather than presence/absence.

Petite frequency is expressed as the percentage of petite colonies relative to
the total scored.

## Scoring pipeline

Colony areas are measured in **Fiji** (Schindelin et al., 2012) and exported as
one CSV per plate. This notebook then:

1. Plots the colony area distribution for each plate
2. **The operator clicks the histogram to set the large/small boundary**
3. Scores colonies either side of that boundary and records the counts
4. Pools the technical replicate plates per biological sample
5. Compares genotypes on per-replicate petite ratios

**Threshold selection is manual and set independently for each plate.** This is
the "semi-automated" element of the pipeline and the central methodological
choice in the analysis, since petite frequency is determined entirely by where
the boundary falls.

Because scoring is interactive, the exported summary tables — not the raw Fiji
CSVs — are the reproducible record. Downstream cells read those tables and run
without the manual step.

## Analysis conventions

- **Unit of replication:** the biological replicate (n = 3), not the colony. Ratios are computed per replicate by pooling counts across that replicate's plates, which weights each plate by the colonies it contributed.
- **Test:** Welch's *t*-test on per-replicate petite ratios. Exact p-values are reported rather than significance stars.
- **Comparisons** are pre-specified by the strain design — each strain against WT, and each deletion against its complemented strains.
- 
## Running this notebook

The scoring cell requires an interactive matplotlib backend (`%matplotlib qt`)
and will not run in a rendered notebook. Set `DATA_DIR` (Fiji CSVs and cached
summaries) and `OUT_DIR` (figures) in the first cell.

**Requires:** `pandas`, `numpy`, `scipy`, `seaborn`, `matplotlib`, plus Fiji for
the upstream colony measurement.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os
import time
import seaborn as sns
import scipy
from scipy import stats
#import pickle5
%matplotlib qt

## 1. Setup and input validation

Imports and a pre-flight check over the colony measurement tables exported from
Fiji.

Each CSV holds the particle measurements for one plate. The loop reads every
file and reports any that contain non-numeric columns, which would indicate a
malformed export or an unexpected column (for example a label column or a
partially written file) before it reaches the analysis.


In [ ]:
DATA_DIR = '.'
folder_path = f'{DATA_DIR}/csv_files/'

for filename in os.listdir(folder_path):
    if filename.endswith('.csv'):
        file_path = os.path.join(folder_path, filename)
        
        try:
            df = pd.read_csv(file_path)
            
            if df.select_dtypes(include=['number']).shape[1] != df.shape[1]:
                print(f"{filename}: Not all columns contain numeric values.")
        
        except Exception as e:
            print(f"{filename}: Error - {str(e)}")

## 2. Interactive threshold selection and petite scoring

For each plate, the colony area distribution is plotted as a histogram and the
operator clicks the point separating the two size classes. Colonies below the
clicked area are scored as petite (small, respiratory-deficient), those above as
grande (large, respiratory-competent).

Metadata — genotype, replicate, condition — is parsed from the filename.
Expected pattern: `<genotype>_<replicate>_<condition>.csv`.

For each plate the output records the petite and grande fractions, the absolute
counts of each, and the total number of colonies scored.

**Threshold selection is manual and set independently per plate.** Petite
frequency depends entirely on where this boundary falls, so the threshold is the
central methodological choice in this analysis.

In [ ]:
plt.ion()

DATA_DIR = '.'
root_Folder = f'{DATA_DIR}/csv_files/'
os.chdir(root_Folder)
files = os.listdir(root_Folder)
files = [x for x in files if x.endswith('.csv')]

summary = pd.DataFrame()
def onclick(event):
    global ix, iy
    ix, iy = event.xdata, event.ydata
    print('x = %d, y = %d'%(ix, iy))

    global coords
    coords.append((ix, iy))

    if len(coords) == 2:
        fig.canvas.mpl_disconnect(cid)

    pass


fig = plt.figure()
ax = fig.add_subplot(111)
for file in files:
    fig.clear()
    df = pd.read_csv(root_Folder + file)
    df['Name'] = file.strip('.csv')
    df['Replicate'] = df['Name'].str.split('_').str[1]
    df['Condition'] = df['Name'].str.split('_').str[2]
    #df['Replicate'] = df['Replicate'].str.split('.', expand=True)[0]
    df['Genotype'] = df['Name'].str.split('_').str[0]
    count, division = np.histogram(df.iloc[:,1], bins=50, )
    x_values = (division[:-1]+(division[2]-division[1])*0.5)
    coords=[]

    df.iloc[:,1].plot.hist(bins= 50, )
    plt.plot(x_values, count)
    plt.title(file)
    fig.canvas.draw()
    
    cid = fig.canvas.mpl_connect('button_press_event', onclick)
    w = plt.waitforbuttonpress() 
    counts = pd.cut(df['Area'], ([0, coords[0][0], df['Area'].max()]), labels=['Petite', 'Grande']).value_counts(normalize=True).to_frame()
    #counts = pd.cut(df['Area'], [0, 0.0001, coords[0][0],1], labels=['tiny', 'petite', 'big']).value_counts(normalize=True).to_frame()
    #counts = pd.cut(df['Area'], np.sort([0, 0.0001, coords[0][0],1]), labels=['tiny', 'petite', 'big']).value_counts(normalize=True).to_frame()

    counts['Name'] = file
    counts['Genotype'] = counts['Name'].str.split('_').str[0]
    #counts['Timepoint'] = counts['Name'].str.split('_').str[1]
    counts['Condition'] = counts['Name'].str.split('_').str[2]
    counts['Replicate'] = counts['Name'].str.split('_').str[1] #+ '_' + counts['Name'].str.split('_').str[2]
    #counts['Replicate'] = counts['Replicate'].str.split('.', expand=True)[0]
    #counts['Replicate'] = counts['Name'].str.split('_').str[2].str.split('.').str[0]
    #counts['Genotype_Replicate'] = counts['Genotype'] + '_' + counts['Timepoint'] + '_' + counts['Concentration'] + '_' + counts['Replicate']
    counts['Genotype_Replicate_Condition'] = counts['Genotype'] + '_' + counts['Replicate'] + '_' + counts['Condition']
    counts['Number of colonies'] = df.shape[0]
    counts['no_petites'] = (pd.cut(df['Area'], [0, coords[0][0], df['Area'].max()], labels=['Petite', 'Grande']) == 'Petite').sum()
    counts['no_grande'] = (pd.cut(df['Area'], [0, coords[0][0], df['Area'].max()], labels=['Petite', 'Grande']) == 'Grande').sum()
    summary = pd.concat([summary, counts])
    summary_2 =summary.reset_index().groupby(['Area', 'Genotype_Replicate_Condition']).mean(numeric_only=True).reset_index()
    

## 3. Export scored results

Writes the per-plate petite/grande summary to CSV. Because threshold selection
is interactive, this file is the durable record of the scoring session — the
downstream analysis reads it rather than repeating the manual step.

In [ ]:
summary_2.to_csv(f'{OUT_DIR}/summary_2.csv')

## 4. Aggregate plates to per-sample petite frequency

Loads the merged scoring results, converts petite and grande proportions back to
absolute colony counts, and sums the technical replicate plates belonging to each
biological sample.

Petite frequency per sample is then the total petite colonies over the total
colonies scored across both plates — pooling counts rather than averaging
proportions, which correctly weights plates by the number of colonies they
contributed.

Metadata is parsed from the composite `genotype_replicate_condition` string:
`genotype_replicate` identifies the biological sample, `plate` the technical
replicate within it.

In [ ]:
summary_2 = pd.read_csv(f'{DATA_DIR}/DATA.csv')
summary_2


### everything for combining csv/dfs for plotting PDH data

In [ ]:
summary_2['no_petites'] = summary_2['Area'] * summary_2['Number of colonies']
summary_2['no_grandes'] = (1 - summary_2['Area']) * summary_2['Number of colonies']
summary_2

In [ ]:
petite_df = summary_2[summary_2['Area'] == 'Petite']
#petite_df['Genotype_Replicate'] = petite_df['Genotype_Replicate'].str.replace('wt', 'wwt')
petite_df = petite_df.rename(columns={'Genotype_Replicate_Condition': 'genotype', 'proportion': 'petite_ratio', 'Number of colonies':'number_of_colonies'})
#petite_df = petite_df.drop(columns=['Unnamed: 0'])

petite_df

In [ ]:
petite_df['genotype_replicate'] = petite_df['genotype'].str.split("_").str[:2].str.join("_")


petite_df['plate'] = petite_df['genotype'].str.split("_").str[2]
petite_df['plate'] = petite_df['plate'].str.replace('.Tif.csv', '')
petite_df['genotype'] = petite_df['genotype'].str.replace('.Tif.csv', '')

petite_df

In [ ]:
sum_plates = pd.DataFrame(columns=['genotype','genotype_replicate',  'number_of_colonies', 'no_petites', 'no_grandes'])

# Iterate through the DataFrame in pairs and calculate sums manually
for i in range(0, len(petite_df)-1, 2):
    row1 = petite_df.iloc[i]
    row2 = petite_df.iloc[i+1]
    
    sum_values = {
        'genotype': row1['genotype'],
        'genotype_replicate': row1['genotype_replicate'],
        'number_of_colonies': row1['number_of_colonies'] + row2['number_of_colonies'],
        'no_petites': row1['no_petites'] + row2['no_petites'],
        'no_grandes': row1['no_grande'] + row2['no_grande']
    }
    
    sum_plates = pd.concat([sum_plates,pd.DataFrame([sum_values])], ignore_index=True)
    sum_plates['petite_ratio'] =  sum_plates['no_petites'] /  sum_plates['number_of_colonies']

sum_plates

In [ ]:
sum_plates.to_pickle(f'{OUT_DIR}/DATA.pkl')

In [ ]:
Nupur2 = ["#88A27D","#b6c9c1","#BD5B28","#FFD30A","#4d6171","#162734"]
sns.color_palette(Nupur2)

## 5. Petite frequency by genotype

Compares petite frequency across a strain panel: wild type, the deletion, and the
deletion complemented with either the wild-type gene or a catalytically inactive
variant. Genotype and biological replicate are parsed from the composite
`genotype_replicate` identifier.

**Comparisons.** Each strain against wild type, and the deletion against each of
its complemented strains. The second set is the informative one: it tests
whether complementation restores respiratory competence, and whether that
restoration requires catalytic activity.

**Test.** Welch's *t*-test (unequal variances) on per-replicate petite ratios,
n = 3 per genotype. The unit of replication is the biological replicate, not the
colony — petite ratios are computed per replicate from pooled plate counts, so
colonies are not treated as independent observations. Exact p-values are
annotated rather than significance stars.

**Plot.** Per-replicate petite ratios as large points coloured by biological
replicate, with a box showing the genotype means.

**Note.** The comparison set is pre-specified by the strain design; no
multiple-testing correction is applied. [TODO: confirm and state in the Methods.]

### latmutants

In [ ]:
latmutants = pd.read_pickle(f'{DATA_DIR}/DATA.pkl')
latmutants

In [ ]:
latmutants['genotype_replicate'] = latmutants['genotype_replicate'].apply(lambda x: x[:-1] + '_' + x[-1])
latmutants['genotype'] = latmutants['genotype_replicate'].str.split("_").str[0] 
latmutants['biological_replicate'] = latmutants['genotype_replicate'].str.split("_").str[1]

latmutants

In [ ]:
# Calculate the mean petite_ratio for each genotype
mean_genotype_latmutants = latmutants.groupby('genotype')['petite_ratio'].mean().reset_index()

# Rename the columns to be more descriptive
mean_genotype_latmutants.columns = ['genotype', 'mean_petite_ratio']
mean_genotype_latmutants

In [ ]:
unique_genotypes = mean_genotype_latmutants['genotype'].unique()

# Format genotypes with double quotes and join them with commas
formatted_genotypes = ', '.join(['"{}"'.format(genotype) for genotype in unique_genotypes])

# Display the result
print(formatted_genotypes)

In [ ]:
# Convert 'petite_ratio' to numeric
latmutants['petite_ratio'] = pd.to_numeric(latmutants['petite_ratio'], errors='coerce')

def get_pvalue(p):
    if p < 0.001:
        return "< 0.001"
    else:
        return f"{p:.3f}"

# Perform t-tests for mutants vs wildtype
wt_data = latmutants[latmutants['genotype'] == 'wt']['petite_ratio']
mutants = ["HuD", "del", "lat"]
p_values_wt = {}  # Store p-values for wt comparisons

for mutant in mutants:
    mutant_data = latmutants[latmutants['genotype'] == mutant]['petite_ratio']
    t_stat, p_value = stats.ttest_ind(mutant_data, wt_data, equal_var=False)
    p_values_wt[mutant] = p_value
    print(f"{mutant} vs wildtype: t-statistic = {t_stat:.4f}, p-value = {p_value:.4f}")

# Perform t-tests for del vs lat and del vs HuD
mut_data = latmutants[latmutants['genotype'] == 'del']['petite_ratio']
mutants_2 = ["HuD", "lat"]
p_values_lat = {}  # Store p-values for pda comparisons

for mutant_2 in mutants_2:
    mutant_data = latmutants[latmutants['genotype'] == mutant_2]['petite_ratio']
    t_stat, p_value = stats.ttest_ind(mutant_data, mut_data, equal_var=False)
    p_values_lat[mutant_2] = p_value
    print(f"{mutant_2} vs lat: t-statistic = {t_stat:.4f}, p-value = {p_value:.4f}")


In [ ]:
# Create the plot
plt.figure()
fig, ax = plt.subplots(figsize=(10,6))
u = sns.color_palette(Nupur2,3)

sns.swarmplot(x='genotype', y='petite_ratio', hue='biological_replicate',
              order=[ "wt", "del", "lat",  "HuD"],
              hue_order=['1','2', '3'],
              data = latmutants, alpha=0.8, size=15,
              edgecolor='k', linewidth=1,palette=u, ax=ax)
sns.boxplot(x='genotype', y='mean_petite_ratio',
               order=[ "wt", "del", "lat",  "HuD"],
                  data = mean_genotype_latmutants, ax=ax)

ax.set_ylim(0, 1.45)



# Add significance bars and p-values for wt vs mutants
for i, mutant in enumerate(["del", "lat",  "HuD"]):
    x1, x2 = 0, 1 + i
    y = 1 + i * 0.09
    h = 0.01
    plt.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c='k')
    plt.text((x1+x2)*0.5, y+h*2, f"P = {get_pvalue(p_values_wt[mutant])}", 
             ha='center', va='bottom', color='k')

# Add significance bars and p-values for pda vs pdawt and pda vs pdaG
for i, mutant_2 in enumerate(["lat",  "HuD"]):
    x1, x2 = 1, 2 + i  
    y = 1.28 + i * 0.09
    h = 0.01
    plt.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c='k')
    plt.text((x1+x2)*0.5, y+h*2, f"P = {get_pvalue(p_values_lat[mutant_2])}", 
             ha='center', va='bottom', color='k')



plt.rcParams['svg.fonttype'] = 'none'

# Uncomment the following line if you want to save the figure
plt.savefig(f'{OUT_DIR}/latmut_37_stats_all.svg')

### double mutants 37°C

In [ ]:
doublemutants_37 =  pd.read_pickle(f'{DATA_DIR}/DATA.pkl')
doublemutants_37

In [ ]:
doublemutants_37['genotype'] = doublemutants_37['genotype_replicate'].str.split("_").str[0] 
doublemutants_37['biological_replicate'] = doublemutants_37['genotype_replicate'].str.split("_").str[1]

doublemutants_37

In [ ]:
# Calculate the mean petite_ratio for each genotype
mean_genotype_doublemutants_37 = doublemutants_37.groupby('genotype')['petite_ratio'].mean().reset_index()

# Rename the columns to be more descriptive
mean_genotype_doublemutants_37.columns = ['genotype', 'mean_petite_ratio']
mean_genotype_doublemutants_37

In [ ]:
unique_genotypes = mean_genotype_doublemutants_37['genotype'].unique()

# Format genotypes with double quotes and join them with commas
formatted_genotypes = ', '.join(['"{}"'.format(genotype) for genotype in unique_genotypes])

# Display the result
print(formatted_genotypes)


In [ ]:
doublemutants_37['petite_ratio'] = pd.to_numeric(doublemutants_37['petite_ratio'], errors='coerce')
def get_pvalue(p):
    if p < 0.001:
        return "< 0.001"
    else:
        return f"{p:.3f}"

# Perform t-tests
wt_data = doublemutants_37[doublemutants_37['genotype'] == 'wt']['petite_ratio']
mutants = ["lat", "pda1",  "lat1pda1c4", "pdb1", "lat1pdb1c1"]
p_values = {}

for mutant in mutants:
    mutant_data = doublemutants_37[doublemutants_37['genotype'] == mutant]['petite_ratio']
    t_stat, p_value = stats.ttest_ind(mutant_data, wt_data, equal_var=False)
    p_values[mutant] = p_value
    print(f"{mutant} vs wildtype: t-statistic = {t_stat:.8f}, p-value = {p_value:.8f}")

In [ ]:
# Create the plot
plt.figure()
fig, ax = plt.subplots(figsize=(10,6))
u = sns.color_palette(Nupur2,3)

sns.swarmplot(x='genotype', y='petite_ratio', hue='biological_replicate',
              order=["wt", "lat", "pda1",  "lat1pda1c4",  "pdb1", "lat1pdb1c1"],
              hue_order=['1','2', '3'],
              data=doublemutants_37, alpha=0.8, size=15,
              edgecolor='k', linewidth=1, palette=u, ax=ax)

sns.boxplot(x='genotype', y='mean_petite_ratio',
            order=["wt", "lat", "pda1",  "lat1pda1c4","pdb1", "lat1pdb1c1"],
            data=mean_genotype_doublemutants_37, ax=ax)

ax.set_ylim(0, 1.45)

# Add significance bars and p-values
for i, mutant in enumerate(["lat", "pda1",  "lat1pda1c4","pdb1", "lat1pdb1c1"]):
    x1, x2 = 0, 1 + i
    y = 1 + i * 0.09
    h = 0.01
    plt.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c='k')
    plt.text((x1+x2)*0.5, y+h*2, f"P = {get_pvalue(p_values[mutant])}", 
             ha='center', va='bottom', color='k')

plt.rcParams['svg.fonttype'] = 'none'

# Uncomment the following line if you want to save the figure
plt.savefig(f'{OUT_DIR}/doublemut_37.svg')

### double mutants 30°C

In [ ]:
doublemutants_30 =  pd.read_pickle(f'{DATA_DIR}/DATA.pkl')
doublemutants_30 


In [ ]:
doublemutants_30['genotype'] = doublemutants_30['genotype_replicate'].str.split("_").str[0] 
doublemutants_30['biological_replicate'] = doublemutants_30['genotype_replicate'].str.split("_").str[1]

doublemutants_30

In [ ]:
# Calculate the mean petite_ratio for each genotype
mean_genotype_doublemutants_30 = doublemutants_30.groupby('genotype')['petite_ratio'].mean().reset_index()

# Rename the columns to be more descriptive
mean_genotype_doublemutants_30.columns = ['genotype', 'mean_petite_ratio']
mean_genotype_doublemutants_30

In [ ]:
unique_genotypes = mean_genotype_doublemutants_30['genotype'].unique()

# Format genotypes with double quotes and join them with commas
formatted_genotypes = ', '.join(['"{}"'.format(genotype) for genotype in unique_genotypes])

# Display the result
print(formatted_genotypes)


In [ ]:
doublemutants_30['petite_ratio'] = pd.to_numeric(doublemutants_30['petite_ratio'], errors='coerce')
def get_pvalue(p):
    if p < 0.001:
        return "< 0.001"
    else:
        return f"{p:.3f}"

# Perform t-tests
wt_data = doublemutants_30[doublemutants_30['genotype'] == 'wt']['petite_ratio']
mutants = ["lat1",  "lat1pda1c4", "lat1pdb1c1",  "pda1", "pdb1"]
p_values = {}

for mutant in mutants:
    mutant_data = doublemutants_30[doublemutants_30['genotype'] == mutant]['petite_ratio']
    t_stat, p_value = stats.ttest_ind(mutant_data, wt_data)
    p_values[mutant] = p_value
    print(f"{mutant} vs wildtype: t-statistic = {t_stat:.4f}, p-value = {p_value:.4f}")

In [ ]:
# Create the plot
plt.figure()
fig, ax = plt.subplots(figsize=(10,6))
u = sns.color_palette(Nupur2,3)

sns.swarmplot(x='genotype', y='petite_ratio', hue='biological_replicate',
              order=["wt", "lat1", "pda1",  "lat1pda1c4",  "pdb1", "lat1pdb1c1"],
              hue_order=['1','2', '3'],
              data=doublemutants_30, alpha=0.8, size=15,
              edgecolor='k', linewidth=1, palette=u, ax=ax)

sns.boxplot(x='genotype', y='mean_petite_ratio',
            order=["wt", "lat1", "pda1",  "lat1pda1c4","pdb1", "lat1pdb1c1"],
            data=mean_genotype_doublemutants_30, ax=ax)

ax.set_ylim(0, 1.45)

# Add significance bars and p-values
for i, mutant in enumerate(["lat1", "pda1",  "lat1pda1c4","pdb1", "lat1pdb1c1"]):
    x1, x2 = 0, 1 + i
    y = 1 + i * 0.09
    h = 0.01
    plt.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c='k')
    plt.text((x1+x2)*0.5, y+h*2, f"P = {get_pvalue(p_values[mutant])}", 
             ha='center', va='bottom', color='k')

plt.rcParams['svg.fonttype'] = 'none'

# Uncomment the following line if you want to save the figure
plt.savefig(f'{OUT_DIR}/doublemut_30.svg')

### deletions mutants 30°C

In [ ]:
del_30 = pd.read_pickle(f'{DATA_DIR}/DATA.pkl')
del_30

In [ ]:
del_30['genotype_replicate'] = del_30['genotype_replicate'].apply(lambda x: x[:-1] + '_' + x[-1])
del_30['genotype'] = del_30['genotype_replicate'].str.split("_").str[0] 
del_30['biological_replicate'] = del_30['genotype_replicate'].str.split("_").str[1]

del_30

In [ ]:
# Calculate the mean petite_ratio for each genotype
mean_genotype_del_30 = del_30.groupby('genotype')['petite_ratio'].mean().reset_index()

# Rename the columns to be more descriptive
mean_genotype_del_30.columns = ['genotype', 'mean_petite_ratio']
mean_genotype_del_30

In [ ]:
unique_genotypes = mean_genotype_del_30['genotype'].unique()

# Format genotypes with double quotes and join them with commas
formatted_genotypes = ', '.join(['"{}"'.format(genotype) for genotype in unique_genotypes])

# Display the result
print(formatted_genotypes)


In [ ]:
del_30['petite_ratio'] = pd.to_numeric(del_30['petite_ratio'], errors='coerce')
def get_pvalue(p):
    if p < 0.001:
        return "< 0.001"
    else:
        return f"{p:.3f}"

# Perform t-tests
wt_data = del_30[del_30['genotype'] == 'wt']['petite_ratio']
mutants = ['lat', 'lpd', 'pda', 'pdb', 'pdx']
p_values = {}

for mutant in mutants:
    mutant_data = del_30[del_30['genotype'] == mutant]['petite_ratio']
    t_stat, p_value = stats.ttest_ind(mutant_data, wt_data, equal_var=False)
    p_values[mutant] = p_value
    print(f"{mutant} vs wildtype: t-statistic = {t_stat:.4f}, p-value = {p_value:.4f}")

In [ ]:
%matplotlib inline
%pylab inline

plt.figure()
fig, ax = plt.subplots(figsize=(10,6))
u = sns.color_palette(Nupur2,3)

sns.swarmplot(x='genotype', y='petite_ratio', hue='biological_replicate',
              order=[ "wt", "pda", "pdb",  "lat",  "pdx"],
              hue_order=['1','2', '3'],
              data = del_30, alpha=0.8, size=15,
              edgecolor='k', linewidth=1,palette=u, ax=ax)
sns.boxplot(x='genotype', y='mean_petite_ratio',
              order=[ "wt", "pda", "pdb",  "lat",  "pdx"],
                  data = mean_genotype_del_30, ax=ax)


#plt.errorbar(range(6), mean_technical_replicates['petite_ratio'], yerr=e['petite_ratio'], fmt='none',capsize=5,ecolor='k' )
#ax.get_legend().remove()
ax.set_ylim(0,1)
#for i,j in enumerate(["pda", "pdb",  "lat",  "pdx"]):
 #  x1, x2, y, h = 0, 1+i, 1+i*0.09, 0.01
  # plt.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c='k')
   #plt.text((x1+x2)*.5, y+h*2, "P = "+get_pvalue(f,j), ha='center', va='bottom', color='k')
plt.rcParams['svg.fonttype'] = 'none'

plt.savefig(f'{OUT_DIR}/deletions_30.svg')

In [ ]:
# Create the plot
plt.figure()
fig, ax = plt.subplots(figsize=(10,6))
u = sns.color_palette(Nupur2,3)

sns.swarmplot(x='genotype', y='petite_ratio', hue='biological_replicate',
              order=["wt", "pda", "pdb", "lat",  "pdx"],
              hue_order=['1','2', '3'],
              data=del_30, alpha=0.8, size=15,
              edgecolor='k', linewidth=1, palette=u, ax=ax)

sns.boxplot(x='genotype', y='mean_petite_ratio',
            order=["wt", "pda", "pdb", "lat",  "pdx"],
            data=mean_genotype_del_30, ax=ax)

ax.set_ylim(0, 1.45)

# Add significance bars and p-values
for i, mutant in enumerate(["pda", "pdb", "lat",  "pdx"]):
    x1, x2 = 0, 1 + i
    y = 1 + i * 0.09
    h = 0.01
    plt.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c='k')
    plt.text((x1+x2)*0.5, y+h*2, f"P = {get_pvalue(p_values[mutant])}", 
             ha='center', va='bottom', color='k')

plt.rcParams['svg.fonttype'] = 'none'

# Uncomment the following line if you want to save the figure
plt.savefig('/Volumes/ag-osman/For_Nupur/Collab_FT_JH/petites/final_figures_3/deletions_30.svg')

### deletions mutants 37°C

In [ ]:
del_37 =  pd.read_pickle(f'{DATA_DIR}/DATA.pkl')
del_37

In [ ]:
del_37['genotype'] = del_37['genotype_replicate'].str.split("_").str[0] 
del_37['biological_replicate'] = del_37['genotype_replicate'].str.split("_").str[1]

del_37

In [ ]:
# Calculate the mean petite_ratio for each genotype
mean_genotype_del_37 = del_37.groupby('genotype')['petite_ratio'].mean().reset_index()

# Rename the columns to be more descriptive
mean_genotype_del_37.columns = ['genotype', 'mean_petite_ratio']
mean_genotype_del_37

In [ ]:
del_37['petite_ratio'] = pd.to_numeric(del_37['petite_ratio'], errors='coerce')
def get_pvalue(p):
    if p < 0.001:
        return "< 0.001"
    else:
        return f"{p:.3f}"

# Perform t-tests
wt_data = del_37[del_37['genotype'] == 'wt']['petite_ratio']
mutants = ['lat1',  'pda1', 'pdb1', 'pdx1']
p_values = {}

for mutant in mutants:
    mutant_data = del_37[del_37['genotype'] == mutant]['petite_ratio']
    t_stat, p_value = stats.ttest_ind(mutant_data, wt_data, equal_var=False)
    p_values[mutant] = p_value
    print(f"{mutant} vs wildtype: t-statistic = {t_stat:.4f}, p-value = {p_value:.4f}")

In [ ]:
%matplotlib inline
# Create the plot
plt.figure()
fig, ax = plt.subplots(figsize=(10,6))
u = sns.color_palette(Nupur2,3)

sns.swarmplot(x='genotype', y='petite_ratio', hue='biological_replicate',
              order=["wt", "pda1", "pdb1", "lat1",  "pdx1"],
              hue_order=['1','2', '3'],
              data=del_37, alpha=0.8, size=15,
              edgecolor='k', linewidth=1, palette=u, ax=ax)

sns.boxplot(x='genotype', y='mean_petite_ratio',
            order=["wt", "pda1", "pdb1", "lat1",  "pdx1"],
            data=mean_genotype_del_37, ax=ax)

ax.set_ylim(0, 1.45)

# Add significance bars and p-values
for i, mutant in enumerate(["pda1", "pdb1", "lat1", "pdx1"]):
    x1, x2 = 0, 1 + i
    y = 1 + i * 0.09
    h = 0.01
    plt.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c='k')
    plt.text((x1+x2)*0.5, y+h*2, f"P = {get_pvalue(p_values[mutant])}", 
             ha='center', va='bottom', color='k')

plt.rcParams['svg.fonttype'] = 'none'

# Uncomment the following line if you want to save the figure
plt.savefig(f'{OUT_DIR}/deletions_37.svg')

### MPC 

#### 30°C

In [ ]:
mpc_30 =  pd.read_pickle(f'{DATA_DIR}/DATA.pkl')

In [ ]:
mpc_30['genotype'] = mpc_30['genotype_replicate'].str.split("_").str[0] 
mpc_30['biological_replicate'] = mpc_30['genotype_replicate'].str.split("_").str[1]

mpc_30

In [ ]:
# Calculate the mean petite_ratio for each genotype
mean_genotype_mpc_30 = mpc_30.groupby('genotype')['petite_ratio'].mean().reset_index()

# Rename the columns to be more descriptive
mean_genotype_mpc_30.columns = ['genotype', 'mean_petite_ratio']
mean_genotype_mpc_30

In [ ]:
mpc_30['petite_ratio'] = pd.to_numeric(mpc_30['petite_ratio'], errors='coerce')
def get_pvalue(p):
    if p < 0.001:
        return "< 0.001"
    else:
        return f"{p:.3f}"

# Perform t-tests
wt_data = mpc_30[mpc_30['genotype'] == 'wt']['petite_ratio']
mutants = ['mpc']
p_values = {}

for mutant in mutants:
    mutant_data = mpc_30[mpc_30['genotype'] == mutant]['petite_ratio']
    t_stat, p_value = stats.ttest_ind(mutant_data, wt_data, equal_var=False)
    p_values[mutant] = p_value
    print(f"{mutant} vs wildtype: t-statistic = {t_stat:.4f}, p-value = {p_value:.4f}")

In [ ]:
# Create the plot
plt.figure()
fig, ax = plt.subplots(figsize=(10,6))
u = sns.color_palette(Nupur2,3)

sns.swarmplot(x='genotype', y='petite_ratio', hue='biological_replicate',
              order=["wt", "mpc"],
              hue_order=['1','2', '3'],
              data=mpc_30, alpha=0.8, size=15,
              edgecolor='k', linewidth=1, palette=u, ax=ax)

sns.boxplot(x='genotype', y='mean_petite_ratio',
             order=["wt", "mpc"],
            data=mean_genotype_mpc_30, ax=ax)

ax.set_ylim(0, 1.45)

# Add significance bars and p-values
for i, mutant in enumerate(['mpc']):
    x1, x2 = 0, 1 + i
    y = 1 + i * 0.09
    h = 0.01
    plt.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c='k')
    plt.text((x1+x2)*0.5, y+h*2, f"P = {get_pvalue(p_values[mutant])}", 
             ha='center', va='bottom', color='k')

plt.rcParams['svg.fonttype'] = 'none'

# Uncomment the following line if you want to save the figure
plt.savefig(f'{OUT_DIR}/mpc_30.svg')

#### 37°C

In [ ]:
mpc_37 =  pd.read_pickle(f'{DATA_DIR}/DATA.pkl')
mpc_37

In [ ]:
mpc_37['genotype'] = mpc_37['genotype_replicate'].str.split("_").str[0] 
mpc_37['biological_replicate'] = mpc_37['genotype_replicate'].str.split("_").str[1]

mpc_37

In [ ]:
# Calculate the mean petite_ratio for each genotype
mean_genotype_mpc_37 = mpc_37.groupby('genotype')['petite_ratio'].mean().reset_index()

# Rename the columns to be more descriptive
mean_genotype_mpc_37.columns = ['genotype', 'mean_petite_ratio']
mean_genotype_mpc_37

In [ ]:
mpc_37['petite_ratio'] = pd.to_numeric(mpc_37['petite_ratio'], errors='coerce')
def get_pvalue(p):
    if p < 0.001:
        return "< 0.001"
    else:
        return f"{p:.3f}"

# Perform t-tests
wt_data = mpc_37[mpc_37['genotype'] == 'wt']['petite_ratio']
mutants = ['mpc']
p_values = {}

for mutant in mutants:
    mutant_data = mpc_37[mpc_37['genotype'] == mutant]['petite_ratio']
    t_stat, p_value = stats.ttest_ind(mutant_data, wt_data, equal_var=False)
    p_values[mutant] = p_value
    print(f"{mutant} vs wildtype: t-statistic = {t_stat:.4f}, p-value = {p_value:.4f}")

In [ ]:
# Create the plot
plt.figure()
fig, ax = plt.subplots(figsize=(10,6))
u = sns.color_palette(Nupur2,3)

sns.swarmplot(x='genotype', y='petite_ratio', hue='biological_replicate',
              order=["wt", "mpc"],
              hue_order=['1','2', '3'],
              data=mpc_37, alpha=0.8, size=15,
              edgecolor='k', linewidth=1, palette=u, ax=ax)

sns.boxplot(x='genotype', y='mean_petite_ratio',
             order=["wt", "mpc"],
            data=mean_genotype_mpc_37, ax=ax)

ax.set_ylim(0, 1.45)

# Add significance bars and p-values
for i, mutant in enumerate(['mpc']):
    x1, x2 = 0, 1 + i
    y = 1 + i * 0.09
    h = 0.01
    plt.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c='k')
    plt.text((x1+x2)*0.5, y+h*2, f"P = {get_pvalue(p_values[mutant])}", 
             ha='center', va='bottom', color='k')

plt.rcParams['svg.fonttype'] = 'none'

# Uncomment the following line if you want to save the figure
plt.savefig(f'{OUT_DIR}/mpc_37.svg')

### PDA Mutants

### 30°C

In [ ]:
pda_mut_30 =  pd.read_pickle(f'{DATA_DIR}/DATA.pkl')
pda_mut_30

In [ ]:
pda_mut_30['genotype'] = pda_mut_30['genotype_replicate'].str.split("_").str[0] 
pda_mut_30['biological_replicate'] = pda_mut_30['genotype_replicate'].str.split("_").str[1]

pda_mut_30

In [ ]:
# Calculate the mean petite_ratio for each genotype
mean_genotype_pda_mut_30 = pda_mut_30.groupby('genotype')['petite_ratio'].mean().reset_index()

# Rename the columns to be more descriptive
mean_genotype_pda_mut_30.columns = ['genotype', 'mean_petite_ratio']
mean_genotype_pda_mut_30

In [ ]:
pda_mut_30['petite_ratio'] = pd.to_numeric(pda_mut_30['petite_ratio'], errors='coerce')
def get_pvalue(p):
    if p < 0.001:
        return "< 0.001"
    else:
        return f"{p:.3f}"

# Perform t-tests
wt_data = pda_mut_30[pda_mut_30['genotype'] == 'wt']['petite_ratio']
mutants = ['pda', 'pdawt', 'pdaG']
p_values = {}

for mutant in mutants:
    mutant_data = pda_mut_30[pda_mut_30['genotype'] == mutant]['petite_ratio']
    t_stat, p_value = stats.ttest_ind(mutant_data, wt_data, equal_var=False)
    p_values[mutant] = p_value
    print(f"{mutant} vs wildtype: t-statistic = {t_stat:.4f}, p-value = {p_value:.4f}")

In [ ]:
pda_mut_30['petite_ratio'] = pd.to_numeric(pda_mut_30['petite_ratio'], errors='coerce')
def get_pvalue(p):
    if p < 0.001:
        return "< 0.001"
    else:
        return f"{p:.3f}"

# Perform t-tests
wt_data = pda_mut_30[pda_mut_30['genotype'] == 'wt']['petite_ratio']
mutants = ['pda', 'pdawt', 'pdaG']
p_values = {}

for mutant in mutants:
    mutant_data = pda_mut_30[pda_mut_30['genotype'] == mutant]['petite_ratio']
    t_stat, p_value = stats.ttest_ind(mutant_data, wt_data)
    p_values[mutant] = p_value
    print(f"{mutant} vs wildtype: t-statistic = {t_stat:.4f}, p-value = {p_value:.4f}")

In [ ]:
%matplotlib inline
%pylab inline
# Create the plot
plt.figure()
fig, ax = plt.subplots(figsize=(10,6))
u = sns.color_palette(Nupur2,3)

sns.swarmplot(x='genotype', y='petite_ratio', hue='biological_replicate',
              order=["wt", 'pda', 'pdawt', 'pdaG'],
              hue_order=['1','2', '3'],
              data=pda_mut_30, alpha=0.8, size=15,
              edgecolor='k', linewidth=1, palette=u, ax=ax)

sns.boxplot(x='genotype', y='mean_petite_ratio',
             order=["wt", 'pda', 'pdawt', 'pdaG'],
            data=mean_genotype_pda_mut_30, ax=ax)

ax.set_ylim(0, 1.45)

# Add significance bars and p-values
for i, mutant in enumerate(['pda', 'pdawt', 'pdaG']):
    x1, x2 = 0, 1 + i
    y = 1 + i * 0.09
    h = 0.01
    plt.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c='k')
    plt.text((x1+x2)*0.5, y+h*2, f"P = {get_pvalue(p_values[mutant])}", 
             ha='center', va='bottom', color='k')

plt.rcParams['svg.fonttype'] = 'none'

# Uncomment the following line if you want to save the figure
plt.savefig(f'{OUT_DIR}/pdamutants_30.svg')

### 37°C

In [ ]:
pda_mut_37 = pd.read_pickle(f'{DATA_DIR}/DATA.pkl')

In [ ]:
pda_mut_37['genotype'] = pda_mut_37['genotype_replicate'].str.split("_").str[0] 
pda_mut_37['biological_replicate'] = pda_mut_37['genotype_replicate'].str.split("_").str[1]

pda_mut_37

In [ ]:
# Calculate the mean petite_ratio for each genotype
mean_genotype_pda_mut_37 = pda_mut_37.groupby('genotype')['petite_ratio'].mean().reset_index()

# Rename the columns to be more descriptive
mean_genotype_pda_mut_37.columns = ['genotype', 'mean_petite_ratio']
mean_genotype_pda_mut_37

In [ ]:
pda_mut_37['petite_ratio'] = pd.to_numeric(pda_mut_37['petite_ratio'], errors='coerce')
def get_pvalue(p):
    if p < 0.001:
        return "< 0.001"
    else:
        return f"{p:.3f}"

# Perform t-tests
wt_data = pda_mut_37[pda_mut_37['genotype'] == 'wt']['petite_ratio']
mutants = ['pda', 'pdawt', 'pdaG']
p_values = {}

for mutant in mutants:
    mutant_data = pda_mut_37[pda_mut_37['genotype'] == mutant]['petite_ratio']
    t_stat, p_value = stats.ttest_ind(mutant_data, wt_data,  equal_var=False)
    p_values[mutant] = p_value
    print(f"{mutant} vs wildtype: t-statistic = {t_stat:.4f}, p-value = {p_value:.4f}")

In [ ]:
%matplotlib inline
%pylab inline
# Create the plot
plt.figure()
fig, ax = plt.subplots(figsize=(10,6))
u = sns.color_palette(Nupur2,3)

sns.swarmplot(x='genotype', y='petite_ratio', hue='biological_replicate',
              order=["wt", 'deltapda', 'pdawt', 'pdaG2'],
              hue_order=['1','2', '3'],
              data=pda_mut_37, alpha=0.8, size=15,
              edgecolor='k', linewidth=1, palette=u, ax=ax)

sns.boxplot(x='genotype', y='mean_petite_ratio',
             order=["wt", 'deltapda', 'pdawt', 'pdaG2'],
            data=mean_genotype_pda_mut_37, ax=ax)

ax.set_ylim(0, 1.45)

# Add significance bars and p-values for wt vs mutants
for i, mutant in enumerate(['deltapda', 'pdawt', 'pdaG2']):
    x1, x2 = 0, 1 + i
    y = 1 + i * 0.09
    h = 0.01
    plt.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c='k')
    plt.text((x1+x2)*0.5, y+h*2, f"P = {get_pvalue(p_values_wt[mutant])}", 
             ha='center', va='bottom', color='k')

# Add significance bars and p-values for pda vs pdawt and pda vs pdaG
for i, mutant_2 in enumerate(['pdawt', 'pdaG2']):
    x1, x2 = 1, 2 + i  # start from pda (index 1) and compare with pdawt (index 2) and pdaG (index 3)
    y = 1.3 + i * 0.09
    h = 0.01
    plt.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c='k')
    plt.text((x1+x2)*0.5, y+h*2, f"P = {get_pvalue(p_values_pda[mutant_2])}", 
             ha='center', va='bottom', color='k')

plt.rcParams['svg.fonttype'] = 'none'

# Uncomment the following line if you want to save the figure
plt.savefig(f'{OUT_DIR}/pdamut_37.svg')

### CAP treatment

In [ ]:
petite_df_CAP = pd.read_pickle(f'{DATA_DIR}/DATA.pkl')

In [ ]:
%matplotlib inline
import itertools
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

# ============================================================
# [Data Prep] Work on a copy so the original df is untouched
# ============================================================
plot_df = petite_df_CAP.copy()



# --- Core Statistical Function ---
def get_ttest(df, g1, g2, condition_val):
    """Welch's t-test between two genotypes under a specific condition."""
    v1 = df.loc[(df["genotype"] == g1) & (df["condition"] == condition_val), "petite_ratio"]
    v2 = df.loc[(df["genotype"] == g2) & (df["condition"] == condition_val), "petite_ratio"]
    if len(v1.dropna()) < 2 or len(v2.dropna()) < 2:
        return np.nan, np.nan
    return stats.ttest_ind(v1, v2, nan_policy="omit", equal_var=False)

def p_to_stars(pval):
    if pd.isna(pval): return "ns"
    if pval < 0.001:  return "***"
    if pval < 0.01:   return "**"
    if pval < 0.05:   return "*"
    return "ns"

def add_p_value_annotation(ax, x1, x2, y, text):
    h = (ax.get_ylim()[1] - ax.get_ylim()[0]) * 0.02
    ax.plot([x1, x1, x2, x2], [y - h, y, y, y - h], lw=1.2, c="black")
    ax.text((x1 + x2) * 0.5, y + (h * 0.2), text, ha="center", va="bottom", color="black", fontsize=10)


# --- Toggle Settings ---
PLOT_P_VALUES = True
PLOT_ONLY_SIGNIFICANT = True

# --- Data Configuration ---
genotype_order = ["wt", "pda", "pdb", "lat", "pdx"]
hue_order = ["-", "+"]

# --- 1. Initialize Figure & Plotting ---
fig, ax = plt.subplots(figsize=(12, 8))

try:
    Nupur_palette = sns.color_palette(Nupur2)
    color_dict = {"-": Nupur_palette[1], "+": Nupur_palette[5]}
except NameError:
    color_dict = {"-": "#999999", "+": "#d62728", }  # fallback

sns.barplot(
    ax=ax, x="genotype", y="petite_ratio", data=plot_df,
    order=genotype_order, estimator=np.mean, errorbar="sd",
    palette=color_dict, hue="condition", hue_order=hue_order, alpha=0.85,
)

sns.swarmplot(
    ax=ax, x="genotype", y="petite_ratio", data=plot_df,
    order=genotype_order, hue="condition", hue_order=hue_order,
    dodge=True, color="black", size=7, edgecolor="black",
    linewidth=0.8, alpha=0.8, legend=False,
)

# --- 2. P-value brackets (genotypes compared under EtBr = "+") ---
current_y = plot_df["petite_ratio"].max() * 1.15
y_shift   = plot_df["petite_ratio"].max() * 0.08

if PLOT_P_VALUES:
    geno_to_idx = {name: i for i, name in enumerate(genotype_order)}
    offsets = np.linspace(-0.2, 0.2, len(hue_order))
    cond_offset = dict(zip(hue_order, offsets))["+"]

    target_cond = "+"
    for g1, g2 in itertools.combinations(genotype_order, 2):
        stat, pval = get_ttest(plot_df, g1, g2, target_cond)
        stars = p_to_stars(pval)
        if PLOT_ONLY_SIGNIFICANT and stars == "ns":
            continue
        idx1 = geno_to_idx[g1] + cond_offset
        idx2 = geno_to_idx[g2] + cond_offset
        add_p_value_annotation(ax, idx1, idx2, current_y, stars)
        current_y += y_shift

# --- 3. Aesthetics ---
ax.set_ylim(0, current_y + (y_shift * 0.5) if PLOT_P_VALUES else None)
ax.set_xlabel("Genotype", fontsize=12)
ax.set_ylabel("Petite Ratio", fontsize=12)
ax.set_title("Petite Ratio upon CAP Stress - cells grown on YPD", fontsize=14, pad=15)

handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[:len(hue_order)], labels[:len(hue_order)],
          title="Condition", loc="upper left", frameon=True)

plt.rcParams["svg.fonttype"] = "none"
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/CAP.svg')